In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
print("Key loaded:", os.getenv("GROQ_API_KEY") is not None)

Key loaded: True


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.choices[0].message.content)

Hello!


In [4]:
import requests

resp = requests.get(
    "https://clinicaltrials.gov/api/v2/studies",
    params={
        "query.cond": "rheumatoid arthritis",
        "filter.overallStatus": "RECRUITING",
        "fields": "NCTId,BriefTitle,EligibilityCriteria,MinimumAge,MaximumAge,Sex,Condition,OverallStatus",
        "pageSize": 10,
    },
)

data = resp.json()
print(len(data["studies"]))
print(data["studies"][0])

10
{'protocolSection': {'identificationModule': {'nctId': 'NCT05365009', 'briefTitle': 'Registry of Autoimmune Interstitial Lung Disease'}, 'statusModule': {'overallStatus': 'RECRUITING'}, 'conditionsModule': {'conditions': ['Interstitial Lung Disease Due to Systemic Disease']}, 'eligibilityModule': {'eligibilityCriteria': 'Inclusion Criteria:\n\n* Age ≥ 18 years old\n* Diagnosis of ILD within the last 5 years according to the criteria of the multidisciplinary team composed of at least one pulmonologist and one rheumatologist, with or without respiratory symptoms.\n* ILD defined by the presence of ground glass opacities and / or peribronchovascular or airspace consolidations and / or reticulations and / or traction bronchiectasis and / or honeycomb on high-resolution computed tomography (HRCT) within the last 12 months 17 .\n* One of the following three criteria (see annex 1):\n\nEstablished or early stage CTD 18-30. IPAF according to ATS / ERS 2015 classification criteria 8 ANCA posit

In [5]:
def parse_study(study):
    protocol = study["protocolSection"]
    return {
        "nct_id": protocol["identificationModule"].get("nctId"),
        "title": protocol["identificationModule"].get("briefTitle"),
        "status": protocol["statusModule"].get("overallStatus"),
        "conditions": protocol.get("conditionsModule", {}).get("conditions", []),
        "eligibility_criteria": protocol.get("eligibilityModule", {}).get("eligibilityCriteria", ""),
        "sex": protocol.get("eligibilityModule", {}).get("sex"),
        "min_age": protocol.get("eligibilityModule", {}).get("minimumAge"),
        "max_age": protocol.get("eligibilityModule", {}).get("maximumAge"),
    }

trials = [parse_study(s) for s in data["studies"]]
trials[0]

{'nct_id': 'NCT05365009',
 'title': 'Registry of Autoimmune Interstitial Lung Disease',
 'status': 'RECRUITING',
 'conditions': ['Interstitial Lung Disease Due to Systemic Disease'],
 'eligibility_criteria': 'Inclusion Criteria:\n\n* Age ≥ 18 years old\n* Diagnosis of ILD within the last 5 years according to the criteria of the multidisciplinary team composed of at least one pulmonologist and one rheumatologist, with or without respiratory symptoms.\n* ILD defined by the presence of ground glass opacities and / or peribronchovascular or airspace consolidations and / or reticulations and / or traction bronchiectasis and / or honeycomb on high-resolution computed tomography (HRCT) within the last 12 months 17 .\n* One of the following three criteria (see annex 1):\n\nEstablished or early stage CTD 18-30. IPAF according to ATS / ERS 2015 classification criteria 8 ANCA positivity by immunofluorescence confirmed by ELISA, with or without systemic vasculitis 31.\n\n* Spirometry performed wit

In [6]:
def fetch_trials(condition, page_size=100):
    resp = requests.get(
        "https://clinicaltrials.gov/api/v2/studies",
        params={
            "query.cond": condition,
            "filter.overallStatus": "RECRUITING",
            "fields": "NCTId,BriefTitle,EligibilityCriteria,MinimumAge,MaximumAge,Sex,Condition,OverallStatus",
            "pageSize": page_size,
        },
    )
    return resp.json()["studies"]

conditions = ["rheumatoid arthritis", "breast cancer"]

all_trials = []
for cond in conditions:
    studies = fetch_trials(cond)
    all_trials.extend([parse_study(s) for s in studies])

print(len(all_trials))

200


In [7]:
from minsearch import Index

index = Index(
    text_fields=["title", "eligibility_criteria", "conditions"],
    keyword_fields=["nct_id"]
)

# minsearch expects conditions as text, not a list — join it
for t in all_trials:
    t["conditions"] = " ".join(t["conditions"]) if t["conditions"] else ""

index.fit(all_trials)

In [8]:
results = index.search("rheumatoid arthritis patients over 45", num_results=3)
for r in results:
    print(r["nct_id"], "-", r["title"])

NCT07045896 - Fucoidan in the Treatment of Active Rheumatoid Arthritis
NCT03770923 - Effect of Some Drugs on Rheumatoid Arithritis Activity
NCT03192267 - Early Rheumatoid Arthritis Lung Disease Study


In [9]:
def build_prompt(query, results):
    context = "\n\n".join(
        f"NCT ID: {r['nct_id']}\nTitle: {r['title']}\nEligibility: {r['eligibility_criteria']}"
        for r in results
    )
    return f"""Answer the question based only on the clinical trial information below.
If the information isn't sufficient to answer, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

def rag(query):
    results = index.search(query, num_results=3)
    prompt = build_prompt(query, results)
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(rag("What trials exist for rheumatoid arthritis patients over 45?"))

**Rheumatoid‑arthritis trials that enroll participants older than 45 years (based on the eligibility criteria provided):**

| NCT ID | Title | Age range permitted | Note on inclusion of > 45‑year‑olds |
|--------|-------|----------------------|--------------------------------------|
| **NCT05961592** | *R‑2487 in Patients With Rheumatoid Arthritis* | 18 – 75 years (inclusive) | The upper limit is 75 y, so anyone age > 45 (up to 75) can be enrolled if they meet the other criteria. |
| **NCT03192267** | *Early Rheumatoid Arthritis Lung Disease Study* | 19 – 90 years | The upper limit is 90 y; thus participants > 45 y are eligible. |
| **NCT07045896** | *Fucoidan in the Treatment of Active Rheumatoid Arthritis* | 18 – 65 years (inclusive) | The upper limit is 65 y, so adults > 45 y (up to 65) can be enrolled. |

**Summary**

All three listed studies accept adults older than 45 years within their respective age windows (up to 75, 90, and 65 years). Therefore, for a rheumatoid‑arthritis pat

In [10]:
import re

def parse_age(age_str):
    if not age_str:
        return None
    match = re.search(r"\d+", age_str)
    return int(match.group()) if match else None

# quick test
print(parse_age("18 Years"), parse_age(None), parse_age("N/A"))

18 None None


In [11]:
def check_eligibility(nct_id, patient_age):
    trial = next((t for t in all_trials if t["nct_id"] == nct_id), None)
    if not trial:
        return {"error": "trial not found"}
    
    min_age = parse_age(trial["min_age"])
    max_age = parse_age(trial["max_age"])
    
    eligible = True
    reasons = []
    if min_age is not None and patient_age < min_age:
        eligible = False
        reasons.append(f"patient age {patient_age} is below minimum age {min_age}")
    if max_age is not None and patient_age > max_age:
        eligible = False
        reasons.append(f"patient age {patient_age} is above maximum age {max_age}")
    
    return {
        "nct_id": nct_id,
        "eligible_by_age": eligible,
        "min_age": min_age,
        "max_age": max_age,
        "reasons": reasons,
    }

# quick test — use a real nct_id from your data
print(check_eligibility(all_trials[0]["nct_id"], 46))

{'nct_id': 'NCT05365009', 'eligible_by_age': True, 'min_age': 18, 'max_age': None, 'reasons': []}


In [12]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_eligibility",
            "description": "Check if a patient of a given age is eligible for a specific clinical trial based on its age requirements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nct_id": {"type": "string", "description": "The NCT ID of the trial"},
                    "patient_age": {"type": "integer", "description": "The patient's age in years"},
                },
                "required": ["nct_id", "patient_age"],
            },
        },
    }
]

In [14]:
def build_agentic_prompt(query, results):
    context = "\n\n".join(
        f"NCT ID: {r['nct_id']}\nTitle: {r['title']}\nEligibility: {r['eligibility_criteria']}"
        for r in results
    )
    return f"""You are a clinical trials assistant. You have access to a tool called
check_eligibility that checks whether a patient of a given age meets a trial's age requirements.

If the question mentions a specific NCT ID and a patient age, use the tool to check eligibility —
don't try to reason about ages yourself.

Otherwise, answer using the trial information below.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

In [18]:
def rag_agentic(query):
    nct_match = re.search(r"NCT\d{8}", query)
    
    if nct_match:
        # specific trial mentioned — skip retrieval, go straight to tool-enabled call
        messages = [{"role": "user", "content": query}]
    else:
        # general question — use retrieval as before
        results = index.search(query, num_results=3)
        prompt = build_agentic_prompt(query, results)
        messages = [{"role": "user", "content": prompt}]
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
    )
    
    msg = response.choices[0].message
    
    if msg.tool_calls:
        messages.append(msg)
        for call in msg.tool_calls:
            import json
            args = json.loads(call.function.arguments)
            result = check_eligibility(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
        final = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
        )
        return final.choices[0].message.content
    
    return msg.content

print(rag_agentic("Is a 70-year-old patient eligible for trial NCT07045896?"))

The trial **NCT07045896** is limited to participants **between 18 and 65 years old**. Since the patient is **70 years old**, they **do not meet the age eligibility criteria** for this study.
